<a href="https://colab.research.google.com/github/JSJeong-me/GPT-Agent/blob/main/06%20Deepagent/MRO_DeepAgents_OpenAI_API_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LangChain Deep Agents × OpenAI API: MRO 업무 분석 통합 실습

**목표**: Custom Tools + `memory=["./AGENTS.md"]` + `skills=["./skills/"]` + 실제 OpenAI API 호출을 Google Colab에서 통합 실행합니다.

- 원본 엑셀 대신 **가상 샘플 5건**을 기본 사용합니다. 실제 업무자료 전송은 회사 정책에 따라 별도 승인이 필요합니다.
- 실습 구성: 환경 설정 → 데이터 → Custom Tools → `AGENTS.md` → Skill → API 연결 → Deep Agent 실행 → Tool 추적 → 평가
- 모델 기본값: 공식 Customization 문서에 예시로 표시된 `openai:gpt-5.5` (계정에 해당 모델 접근권한이 없으면 실제 사용 가능한 Tool Calling 모델로 변경).
- **주의**: 이 노트북에는 API 비용이 발생하는 셀이 있습니다. API 호출 셀은 실행 의사가 있을 때 실행하세요.

**참고:** [Deep Agents Quickstart](https://docs.langchain.com/oss/python/deepagents/quickstart), [Customization](https://docs.langchain.com/oss/python/deepagents/customization), [Memory](https://docs.langchain.com/oss/python/deepagents/memory), [Skills](https://docs.langchain.com/oss/python/deepagents/skills).

## 1. Colab 환경 준비

Deep Agents, LangChain OpenAI 통합 및 테이블 분석용 Pandas를 설치합니다. Colab 재시작 요구가 나오면 런타임을 재시작하고 이후 셀부터 다시 실행하세요.

In [1]:
%pip -q install -U deepagents langchain langchain-openai langgraph pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.2.0 which is incompatible.


### Cell 2. API Key 읽기

Colab 왼쪽 열쇠 모양 **Secrets**에 `OPENAI_API_KEY`를 저장하는 방법을 권장합니다. Colab 외 환경에서는 암호 입력을 사용합니다. 키는 출력하거나 저장하지 않습니다.

In [2]:
import os, getpass
try:
    from google.colab import userdata
    saved = userdata.get("OPENAI_API_KEY")
    if saved:
        os.environ["OPENAI_API_KEY"] = saved
except (ImportError, KeyError, Exception):
    pass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")
assert os.environ["OPENAI_API_KEY"].strip(), "API key가 비어 있습니다."
print("API Key 로딩 완료 (비밀값 미출력)")

API Key 로딩 완료 (비밀값 미출력)


### Cell 3. 모델 및 작업 폴더

모델 ID는 변경 가능합니다. `gpt-5.5`는 문서 예시이며, 실제 접근 가능 여부는 사용자 계정 및 API 설정에 따라 다릅니다. Colab `/content`는 세션 종료 시 사라질 수 있습니다.

In [3]:
from pathlib import Path
MODEL_ID = "openai:gpt-5.5"  # 접근 불가 시 계정에서 사용할 수 있는 Tool Calling 모델로 변경
PROJECT = Path("/content/mro-deepagents-training") if Path("/content").exists() else Path.cwd() / "mro-deepagents-training"
PROJECT.mkdir(parents=True, exist_ok=True)
(PROJECT / "skills" / "mro-analysis").mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT)
print("PROJECT:", PROJECT)
print("MODEL_ID:", MODEL_ID)

PROJECT: /content/mro-deepagents-training
MODEL_ID: openai:gpt-5.5


## 2. MRO 가상 업무 데이터

기업 Excel의 주요 의미 구조(접수번호, 시스템, 요구사항, 처리상태, 조치내용)를 모방합니다. 아래 예제는 **교육용 가상 사례**이며 실제 2026년 7월 요구사항 원본 데이터가 아닙니다.

In [4]:
import pandas as pd
sample = [
    {"접수번호":"DEMO-001","시스템":"ERP","요구사항":"인사 평가 화면 이동 조건 수정","처리상태":"처리완료","조치내용":"화면 이동 조건과 경로 수정"},
    {"접수번호":"DEMO-002","시스템":"AS서비스플랫폼","요구사항":"AS 처리예정일 D-2 안내 메일 자동 발송","처리상태":"배포승인","조치내용":"메일 발송 스케줄과 대상 조건 구현"},
    {"접수번호":"DEMO-003","시스템":"ERP","요구사항":"세금계산서 품목명 표기 수정","처리상태":"처리완료","조치내용":"품목명 매핑 수정"},
    {"접수번호":"DEMO-004","시스템":"RPA","요구사항":"월간 처리현황 자동 집계","처리상태":"접수","조치내용":""},
    {"접수번호":"DEMO-005","시스템":"AS서비스플랫폼","요구사항":"담당자 변경 시 알림 메일 발송","처리상태":"검토중","조치내용":"알림 정책 검토 중"},
]
DATA_FILE = PROJECT / "mro_demo.csv"
pd.DataFrame(sample).to_csv(DATA_FILE, index=False, encoding="utf-8-sig")
df = pd.read_csv(DATA_FILE).fillna("")
display(df)
assert df["접수번호"].is_unique

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


,접수번호,시스템,요구사항,처리상태,조치내용
0,DEMO-001,ERP,인사 평가 화면 이동 조건 수정,처리완료,화면 이동 조건과 경로 수정
1,DEMO-002,AS서비스플랫폼,AS 처리예정일 D-2 안내 메일 자동 발송,배포승인,메일 발송 스케줄과 대상 조건 구현
2,DEMO-003,ERP,세금계산서 품목명 표기 수정,처리완료,품목명 매핑 수정
3,DEMO-004,RPA,월간 처리현황 자동 집계,접수,
4,DEMO-005,AS서비스플랫폼,담당자 변경 시 알림 메일 발송,검토중,알림 정책 검토 중


## 3. Custom Tools 구현

`@tool` 데코레이터로 LLM이 호출 가능한 함수를 생성합니다. 결과는 구조화 JSON 문자열로 반환하며 Tool 출력이 곧 원본 근거입니다. 문자열 검색은 정규식이 아닌 리터럴 부분 일치 방식입니다.

In [5]:
import json
from langchain_core.tools import tool

def load_mro():
    return pd.read_csv(DATA_FILE, dtype=str).fillna("")

@tool
def search_mro_requests(keyword: str) -> str:
    """MRO 요구사항과 조치내용에서 사용자 키워드를 검색한다.

    Args:
        keyword: 검색할 키워드 (예: 메일, ERP).
    """
    k = keyword.strip()
    if not k:
        return json.dumps({"error":"검색어가 비어 있습니다."}, ensure_ascii=False)
    data = load_mro()
    columns = ["요구사항", "조치내용", "시스템"]
    mask = pd.Series(False, index=data.index)
    for col in columns:
        mask |= data[col].str.contains(k, case=False, regex=False, na=False)
    return json.dumps({"keyword": k, "count":int(mask.sum()), "records":data.loc[mask].to_dict("records")}, ensure_ascii=False)

@tool
def get_mro_request(receipt_id: str) -> str:
    """접수번호로 특정 MRO 요구사항의 상세 처리 이력을 조회한다.

    Args:
        receipt_id: 접수번호 문자열 (예: DEMO-002).
    """
    data = load_mro()
    matched = data.loc[data["접수번호"] == receipt_id.strip()]
    return json.dumps({"receipt_id":receipt_id, "records":matched.to_dict("records")}, ensure_ascii=False)

@tool
def count_mro_by_status() -> str:
    """MRO 요구사항을 처리상태별로 정확하게 집계한다. 통계 질의에 사용한다."""
    data = load_mro()
    counts = data["처리상태"].value_counts(dropna=False).to_dict()
    return json.dumps({"total":len(data), "by_status":counts}, ensure_ascii=False)

TOOLS = [search_mro_requests, get_mro_request, count_mro_by_status]
print([t.name for t in TOOLS])

['search_mro_requests', 'get_mro_request', 'count_mro_by_status']


### Cell 6. API 비용 없는 Tool 단위 테스트

Agent에 연결하기 전 Tool 함수 출력값을 검증합니다. 실제 검색과 집계는 로컬 Pandas가 수행합니다.

In [6]:
search_out = json.loads(search_mro_requests.invoke({"keyword":"메일"}))
status_out = json.loads(count_mro_by_status.invoke({}))
request_out = json.loads(get_mro_request.invoke({"receipt_id":"DEMO-002"}))
print(json.dumps(search_out, indent=2, ensure_ascii=False))
print(json.dumps(status_out, indent=2, ensure_ascii=False))
assert search_out["count"] == 2
assert status_out["total"] == 5
assert status_out["by_status"]["처리완료"] == 2
assert len(request_out["records"]) == 1
print("PASS: 세 가지 Custom Tool의 로컬 검증")

{
  "keyword": "메일",
  "count": 2,
  "records": [
    {
      "접수번호": "DEMO-002",
      "시스템": "AS서비스플랫폼",
      "요구사항": "AS 처리예정일 D-2 안내 메일 자동 발송",
      "처리상태": "배포승인",
      "조치내용": "메일 발송 스케줄과 대상 조건 구현"
    },
    {
      "접수번호": "DEMO-005",
      "시스템": "AS서비스플랫폼",
      "요구사항": "담당자 변경 시 알림 메일 발송",
      "처리상태": "검토중",
      "조치내용": "알림 정책 검토 중"
    }
  ]
}
{
  "total": 5,
  "by_status": {
    "처리완료": 2,
    "배포승인": 1,
    "접수": 1,
    "검토중": 1
  }
}
PASS: 세 가지 Custom Tool의 로컬 검증


## 4. `AGENTS.md` 생성 (Memory)

이 파일은 모든 질문에 상시 적용할 분석 원칙입니다. 업무 규칙과 출력 형식을 기록하며, 대화 저장소의 대체물은 아닙니다.

In [7]:
MEMORY = """# MRO Analysis Agent — Always-on Instructions

## Purpose
가상의 MRO 요구사항 데이터로 접수 및 처리이력을 분석한다.

## Rules
- 반드시 Custom Tool 조회 결과를 근거로 답한다.
- 건수는 추정하지 말고 count_mro_by_status를 실행한다.
- 처리완료와 배포승인을 서로 다른 상태로 취급한다.
- 특정 사례를 설명할 때 접수번호를 명시한다.
- Tool 결과에 없는 사실은 추론으로 만들어내지 않는다.
- 데이터는 예제용 가상 데이터이며 실제 기업 실적이라 표현하지 않는다.
- 민감정보와 API 키를 노출하지 않는다.

## Response
한국어로 간결하게 답변한다. 검색 근거와 주의사항을 표시한다.
"""
(PROJECT / "AGENTS.md").write_text(MEMORY, encoding="utf-8")
assert (PROJECT / "AGENTS.md").exists()
print((PROJECT / "AGENTS.md").read_text(encoding="utf-8"))

# MRO Analysis Agent — Always-on Instructions

## Purpose
가상의 MRO 요구사항 데이터로 접수 및 처리이력을 분석한다.

## Rules
- 반드시 Custom Tool 조회 결과를 근거로 답한다.
- 건수는 추정하지 말고 count_mro_by_status를 실행한다.
- 처리완료와 배포승인을 서로 다른 상태로 취급한다.
- 특정 사례를 설명할 때 접수번호를 명시한다.
- Tool 결과에 없는 사실은 추론으로 만들어내지 않는다.
- 데이터는 예제용 가상 데이터이며 실제 기업 실적이라 표현하지 않는다.
- 민감정보와 API 키를 노출하지 않는다.

## Response
한국어로 간결하게 답변한다. 검색 근거와 주의사항을 표시한다.



## 5. `skills/` 생성 (Skill)

`SKILL.md`의 YAML Frontmatter에 이름과 설명을 작성합니다. Agent는 메타데이터를 이용해 관련 Skill을 찾고 필요 시 상세 지침을 읽습니다. Skill은 Tool 코드를 실행하는 대체물이 아니라 실행 절차를 안내합니다.

In [8]:
SKILL_TEXT = """---
name: mro-analysis
description: Use for MRO requirement searches, status summaries, request history analysis, and evidence-based reports.
---
# MRO 업무 분석 절차

1. 사용자의 질문이 개별 검색인지, 접수번호 상세 조회인지, 통계인지 구분한다.
2. 개별 검색은 `search_mro_requests`, 접수번호 조회는 `get_mro_request`, 집계는 `count_mro_by_status`를 사용한다.
3. 결과를 검증한다. 검색 결과가 없으면 없다고 말한다.
4. 모든 수치는 Tool 결과로부터 기재하고, 식별 가능한 접수번호를 근거로 표시한다.
5. 보고서는 '분석 목표 / 검색 및 집계 결과 / 주요 관찰 / 근거 접수번호 / 한계'로 구성한다.
6. 처리완료는 배포완료와 동일하지 않다. 조치내용이 빈 경우 완료한 것으로 간주하지 않는다.
"""
SKILL_FILE = PROJECT / "skills" / "mro-analysis" / "SKILL.md"
SKILL_FILE.write_text(SKILL_TEXT, encoding="utf-8")
assert SKILL_FILE.exists()
print("Skill:", SKILL_FILE)
print(SKILL_FILE.read_text(encoding="utf-8")[:420])

Skill: /content/mro-deepagents-training/skills/mro-analysis/SKILL.md
---
name: mro-analysis
description: Use for MRO requirement searches, status summaries, request history analysis, and evidence-based reports.
---
# MRO 업무 분석 절차

1. 사용자의 질문이 개별 검색인지, 접수번호 상세 조회인지, 통계인지 구분한다.
2. 개별 검색은 `search_mro_requests`, 접수번호 조회는 `get_mro_request`, 집계는 `count_mro_by_status`를 사용한다.
3. 결과를 검증한다. 검색 결과가 없으면 없다고 말한다.
4. 모든 수치는 Tool 결과로부터 기재하고, 식별 가능한 접수번호를 근거로 표시한다.
5. 보고서는 '분석 목표 / 검색 및 집계 결과 / 주요 관찰


## 6. OpenAI API 연결 점검

여기서부터 **실제 API 요청 및 과금**이 발생합니다. 테스트는 본 Agent와 동일한 모델 계열을 사용해 계정 접근을 확인합니다. 모델 ID는 `openai:` 접두사를 제외한 부분을 사용합니다. 실패하면 API Key, 잔액, 모델 접근 권한을 확인하세요.

In [9]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
OPENAI_MODEL = MODEL_ID.removeprefix("openai:")
llm = ChatOpenAI(model=OPENAI_MODEL, max_retries=1, timeout=90)
probe = llm.invoke([HumanMessage(content="연결 테스트입니다. 'API 연결 성공'만 출력하세요.")])
print("OpenAI 응답:", probe.content)
assert probe.content, "모델 응답이 비어 있습니다."

OpenAI 응답: API 연결 성공


## 7. Deep Agent 구성: Tools + Memory + Skills + Backend

- `FilesystemBackend(root_dir=..., virtual_mode=True)`를 사용해 프로젝트의 `AGENTS.md` 및 `skills/`를 읽습니다.
- `memory=["./AGENTS.md"]`, `skills=["./skills/"]`는 공식 문서의 선언식을 따릅니다.
- `MemorySaver`는 **Colab 런타임 내** 동일 `thread_id`의 대화 상태를 연결하기 위한 별도 Checkpointer입니다. 세션을 넘어가는 영구 저장은 아닙니다.
- FilesystemBackend는 Agent에게 프로젝트 폴더 파일 접근을 허용합니다. 실무에서는 별도의 격리·승인 정책이 필요합니다.

In [10]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langgraph.checkpoint.memory import MemorySaver

backend = FilesystemBackend(root_dir=str(PROJECT), virtual_mode=True)
checkpointer = MemorySaver()
agent = create_deep_agent(
    model=MODEL_ID,
    tools=TOOLS,
    memory=["./AGENTS.md"],
    skills=["./skills/"],
    backend=backend,
    checkpointer=checkpointer,
    system_prompt=(
        "당신은 MRO 업무 분석 Deep Agent입니다. "
        "MRO 데이터 관련 질문에는 제공된 Custom Tool을 사용하고, "
        "업무 분석에는 mro-analysis Skill을 읽어 적용하세요. "
        "AGENTS.md의 원칙을 따르고 도구 응답만 근거로 서술하세요."
    ),
)
print("Deep Agent 생성 완료")

Deep Agent 생성 완료


## 8. 실제 OpenAI API를 통한 통합 Agent 실행

아래 질의는 **집계 + 키워드 검색 + 개별 레코드 조회**를 요구합니다. Agent는 자체 판단으로 Tool을 호출하므로, 호출 횟수와 순서가 매번 같을 필요는 없습니다. `recursion_limit`는 무한 반복을 방지합니다.

In [11]:
QUESTION = """교육용 가상 MRO 데이터로 다음을 수행하세요.
1) 처리상태별 건수와 전체 건수를 정확히 집계하세요.
2) '메일' 관련 요구사항을 검색하고 접수번호와 상태를 제시하세요.
3) DEMO-002의 조치내용을 조회하세요.
4) mro-analysis Skill의 보고서 절차를 따르고, 없는 사실은 지어내지 마세요.
"""
result = agent.invoke(
    {"messages":[{"role":"user","content":QUESTION}]},
    config={"configurable":{"thread_id":"mro-deepagents-demo-001"}, "recursion_limit":35},
)
last = result["messages"][-1]
print("=== Agent 최종 응답 ===")
print(last.content)

=== Agent 최종 응답 ===
[{'type': 'text', 'text': '## 분석 목표\n교육용 가상 MRO 데이터에서 처리상태별 건수 집계, ‘메일’ 관련 요구사항 검색, DEMO-002 조치내용 조회를 수행했습니다.\n\n## 검색 및 집계 결과\n\n### 1) 처리상태별 건수 및 전체 건수\n| 처리상태 | 건수 |\n|---|---:|\n| 처리완료 | 2 |\n| 배포승인 | 1 |\n| 접수 | 1 |\n| 검토중 | 1 |\n| **전체** | **5** |\n\n※ 처리완료와 배포승인은 별도 상태로 구분했습니다.\n\n### 2) ‘메일’ 관련 요구사항 검색 결과\n| 접수번호 | 처리상태 |\n|---|---|\n| DEMO-002 | 배포승인 |\n| DEMO-005 | 검토중 |\n\n총 2건이 검색되었습니다.\n\n### 3) DEMO-002 조치내용\n- 접수번호: DEMO-002\n- 조치내용: **메일 발송 스케줄과 대상 조건 구현**\n\n## 주요 관찰\n- ‘메일’ 관련 요구사항은 DEMO-002, DEMO-005 총 2건입니다.\n- DEMO-002는 상태가 **배포승인**이며, 조치내용이 확인됩니다.\n- DEMO-005는 상태가 **검토중**입니다.\n\n## 근거 접수번호\n- DEMO-002: AS 처리예정일 D-2 안내 메일 자동 발송\n- DEMO-005: 담당자 변경 시 알림 메일 발송\n\n## 한계\n- 위 결과는 제공된 Custom Tool 조회 결과에 근거한 교육용 가상 데이터 분석입니다.\n- Tool 결과에 없는 일정, 담당자, 실제 운영 성과 등은 판단하거나 추정하지 않았습니다.', 'annotations': [], 'id': 'msg_00c60deb1a03d754006af560d46a2887d2bc5f41fb0caeb511', 'phase': 'final_answer'}]


## 9. Tool Calling Trace 분석

AIMessage의 `tool_calls`와 ToolMessage를 출력합니다. 실제 어떤 Tool이 실행되었는지 감사 로그 형태로 점검할 수 있습니다. Skill 로딩 여부는 `read_file` 등 파일 읽기 도구 호출이 포함되는지 확인하세요.

In [12]:
from langchain_core.messages import AIMessage, ToolMessage
calls, tool_results = [], []
for msg in result["messages"]:
    if isinstance(msg, AIMessage):
        for call in getattr(msg, "tool_calls", []) or []:
            calls.append(call)
            print("CALL:", call.get("name"), "args=", call.get("args"))
    elif isinstance(msg, ToolMessage):
        tool_results.append(msg)
        print("RETURN:", msg.name, str(msg.content)[:350])
print("총 도구 호출:", len(calls), "/ 도구 결과:", len(tool_results))
print("호출 도구:", sorted({x.get("name", "") for x in calls}))

CALL: read_file args= {'file_path': '/skills/mro-analysis/SKILL.md', 'offset': 0, 'limit': 1000}
RETURN: read_file @@ lines 1-12 of 12 @@
---
name: mro-analysis
description: Use for MRO requirement searches, status summaries, request history analysis, and evidence-based reports.
---
# MRO 업무 분석 절차

1. 사용자의 질문이 개별 검색인지, 접수번호 상세 조회인지, 통계인지 구분한다.
2. 개별 검색은 `search_mro_requests`, 접수번호 조회는 `get_mro_request`, 집계는 `count_mro_by_status`를 사용한다.
3. 결과를 검증한다. 검색 결과가 없으면 없
CALL: count_mro_by_status args= {}
CALL: search_mro_requests args= {'keyword': '메일'}
CALL: get_mro_request args= {'receipt_id': 'DEMO-002'}
RETURN: count_mro_by_status {"total": 5, "by_status": {"처리완료": 2, "배포승인": 1, "접수": 1, "검토중": 1}}
RETURN: search_mro_requests {"keyword": "메일", "count": 2, "records": [{"접수번호": "DEMO-002", "시스템": "AS서비스플랫폼", "요구사항": "AS 처리예정일 D-2 안내 메일 자동 발송", "처리상태": "배포승인", "조치내용": "메일 발송 스케줄과 대상 조건 구현"}, {"접수번호": "DEMO-005", "시스템": "AS서비스플랫폼", "요구사항": "담당자 변경 시 알림 메일 발송", "처리상태": "검토중", "조치내용": "알림 정책 검토 중

## 10. Ground Truth와 응답 비교

수치 정확성은 Pandas와 Custom Tool 결과를 기준으로 검증하고, 응답에 근거 접수번호가 포함됐는지 확인합니다. 다음 출력은 답변 전체의 의미적 정확성을 자동 보장하지 않습니다.

In [13]:
truth = load_mro()["처리상태"].value_counts().to_dict()
tool_truth = json.loads(count_mro_by_status.invoke({}))
assert truth == tool_truth["by_status"]
assert tool_truth["total"] == 5
used_tools = {c.get("name") for c in calls}
print("Ground Truth:", truth)
print("호출 확인:", used_tools)
print("집계 Tool 호출:", "count_mro_by_status" in used_tools)
print("검색 Tool 호출:", "search_mro_requests" in used_tools)
print("상세조회 Tool 호출:", "get_mro_request" in used_tools)
print("최종답변 DEMO-002 언급:", "DEMO-002" in str(last.content))
print("※ Tool 호출 여부는 모델의 선택에 따라 달라질 수 있으며 미호출 시 질문을 더 명시적으로 수정하세요.")

Ground Truth: {'처리완료': 2, '배포승인': 1, '접수': 1, '검토중': 1}
호출 확인: {'count_mro_by_status', 'read_file', 'search_mro_requests', 'get_mro_request'}
집계 Tool 호출: True
검색 Tool 호출: True
상세조회 Tool 호출: True
최종답변 DEMO-002 언급: True
※ Tool 호출 여부는 모델의 선택에 따라 달라질 수 있으며 미호출 시 질문을 더 명시적으로 수정하세요.


## 11. 동일 세션 후속 질의 (선택)

앞서 사용한 `thread_id`를 재사용하면 `MemorySaver`가 같은 Colab 세션 안에서 대화 상태를 이어갈 수 있습니다. `AGENTS.md`와 Checkpointer는 서로 다른 개념입니다.

In [14]:
followup = agent.invoke(
    {"messages":[{"role":"user","content":"앞서 찾은 메일 관련 요구사항 중 아직 처리완료 상태가 아닌 접수번호를 다시 알려주세요. 근거를 확인하세요."}]},
    config={"configurable":{"thread_id":"mro-deepagents-demo-001"}, "recursion_limit":35},
)
print(followup["messages"][-1].content)

[{'type': 'text', 'text': '메일 관련 요구사항 2건 중 **처리완료 상태가 아닌 접수번호**는 다음과 같습니다.\n\n| 접수번호 | 처리상태 | 근거 |\n|---|---|---|\n| DEMO-002 | 배포승인 | ‘메일’ 검색 결과에서 처리상태가 처리완료가 아님 |\n| DEMO-005 | 검토중 | ‘메일’ 검색 결과에서 처리상태가 처리완료가 아님 |\n\n주의: **배포승인**은 **처리완료**와 다른 상태로 취급했습니다.', 'annotations': [], 'id': 'msg_00c60deb1a03d754006af5615af85887d2a58761a7754d60b4', 'phase': 'final_answer'}]


## 12. 실무 확장 — 실제 Excel의 안전한 적용

이번 노트북은 가상 데이터 5건만 다룹니다. 기존 `MRO_Excel_LangChain_Hybrid_RAG_Colab.ipynb`의 정규화된 DataFrame을 CSV로 내보내고 Tool의 데이터 소스만 교체할 수 있습니다.

1. `.xls`의 다중 행 헤더·하위 작업·날짜·결재 필드를 정규화합니다.
2. `load_mro()`를 해당 정규화 파일을 읽도록 수정합니다.
3. 검색 대상 컬럼 및 접수번호 키를 실제 스키마에 맞게 조정합니다.
4. `semantic_search_mro` Custom Tool을 추가해 Chroma 검색을 결합하면 Hybrid RAG가 됩니다.
5. **기업 승인 전에는 실제 요구사항/고객/개인정보를 외부 API로 보내지 마세요.** 검색 Tool의 반환값도 모델에게 전송됩니다.

### 수강생 평가 체크리스트
- [ ] 키 보안 설정 및 OpenAI 연결 성공
- [ ] Custom Tool 3개 단독 PASS
- [ ] `AGENTS.md` 파일 존재 확인
- [ ] `SKILL.md` YAML Frontmatter 존재 확인
- [ ] Deep Agent 응답 및 도구 호출 로그 확인
- [ ] 상태별 건수와 Pandas Ground Truth 일치
- [ ] 실제 기업 데이터를 사용하지 않았는지 확인

### 참고 문서
- https://docs.langchain.com/oss/python/deepagents/quickstart
- https://docs.langchain.com/oss/python/deepagents/customization
- https://docs.langchain.com/oss/python/deepagents/memory
- https://docs.langchain.com/oss/python/deepagents/skills